In [2]:
import sys
from pathlib import Path

BSF_ROOT = Path("/ebs/home/lily_switch_box/buildstock-fetch")
RDP_ROOT = Path("/ebs/home/lily_switch_box/rate-design-platform")

# Both repos have a top-level `utils/` package — only one can be cached as `utils`.
# Put buildstock-fetch first so `from utils import ev_utils` works.
# Import rate-design-platform's utils.loads via importlib in the next cell.
for root in (RDP_ROOT, BSF_ROOT):
    root_str = str(root)
    if root_str in sys.path:
        sys.path.remove(root_str)
sys.path.insert(0, str(BSF_ROOT))
sys.path.insert(1, str(RDP_ROOT))


In [ ]:
from pathlib import Path

import polars as pl

from data.eia.hourly_loads.eia_region_config import get_aws_storage_options

# ── Edit these ──────────────────────────────────────────────────────────────
STATE = "MD"
UPGRADE = "00"          # try "02" later for the heat-pump upgrade
USE_LOCAL = True        # False → read from s3://data.sb/...
USE_SB = True           # False → raw NREL metadata.parquet
SEED = 42
# ────────────────────────────────────────────────────────────────────────────

RELEASE = "res_2024_amy2018_2"
RELEASE_SB = f"{RELEASE}_sb"
PATH_LOCAL = Path("/ebs/data/nrel/resstock")
PATH_S3 = "s3://data.sb/nrel/resstock"

release_name = RELEASE_SB if USE_SB else RELEASE
resstock_base = (
    PATH_LOCAL / release_name if USE_LOCAL else f"{PATH_S3}/{release_name}"
)
storage_options = get_aws_storage_options() if not USE_LOCAL else None
meta_filename = "metadata-sb.parquet" if USE_SB else "metadata.parquet"

path_metadata = (
    f"{resstock_base}/metadata/state={STATE}/upgrade={UPGRADE}/{meta_filename}"
)
path_loads_dir = (
    f"{resstock_base}/load_curve_hourly/state={STATE}/upgrade={UPGRADE}"
)

print(f"Metadata:  {path_metadata}")
print(f"Loads dir: {path_loads_dir}")

Metadata:  /ebs/data/nrel/resstock/res_2024_amy2018_2_sb/metadata/state=MD/upgrade=00/metadata-sb.parquet
Loads dir: /ebs/data/nrel/resstock/res_2024_amy2018_2_sb/load_curve_hourly/state=MD/upgrade=00


In [5]:
# load metadata
kwargs = {"storage_options": storage_options} if storage_options else {}
metadata = pl.read_parquet(path_metadata, **kwargs)

print(f"{metadata.height:,} buildings × {metadata.width} columns")
metadata.head(3)

9,996 buildings × 188 columns


upgrade,weight,in.sqft,in.representative_income,in.ahs_region,in.aiannh_area,in.area_median_income,in.ashrae_iecc_climate_zone_2004,in.ashrae_iecc_climate_zone_2004_2_a_split,in.bathroom_spot_vent_hour,in.battery,in.bedrooms,in.building_america_climate_zone,in.cec_climate_zone,in.ceiling_fan,in.census_division,in.census_division_recs,in.census_region,in.city,in.clothes_dryer,in.clothes_dryer_usage_level,in.clothes_washer,in.clothes_washer_presence,in.clothes_washer_usage_level,in.cooking_range,in.cooking_range_usage_level,in.cooling_setpoint,in.cooling_setpoint_has_offset,in.cooling_setpoint_offset_magnitude,in.cooling_setpoint_offset_period,in.corridor,in.county,in.county_and_puma,in.county_name,in.dehumidifier,in.dishwasher,in.dishwasher_usage_level,…,in.solar_hot_water,in.state,in.tenure,in.units_represented,in.usage_level,in.utility_bill_electricity_fixed_charges,in.utility_bill_electricity_marginal_rates,in.utility_bill_fuel_oil_fixed_charges,in.utility_bill_fuel_oil_marginal_rates,in.utility_bill_natural_gas_fixed_charges,in.utility_bill_natural_gas_marginal_rates,in.utility_bill_propane_fixed_charges,in.utility_bill_propane_marginal_rates,in.utility_bill_scenario_names,in.utility_bill_simple_filepaths,in.vacancy_status,in.vintage,in.vintage_acs,in.water_heater_efficiency,in.water_heater_fuel,in.water_heater_in_unit,in.water_heater_location,in.weather_file_city,in.weather_file_latitude,in.weather_file_longitude,in.window_areas,in.windows,bldg_id,postprocess_group.has_hp,postprocess_group.heating_type,postprocess_group.heating_type_v2,heats_with_electricity,heats_with_natgas,heats_with_oil,heats_with_propane,has_natgas_connection,mf_non_hvac_electricity_adjusted
i64,f64,i64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,i64,str,i64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,f64,f64,str,str,i64,bool,str,str,bool,bool,bool,bool,bool,bool
0,252.301639,1682,108692.0,"""Non-CBSA South Atlantic""","""No""","""120-150%""","""4A""","""4A""","""Hour6""","""None""","""3""","""Mixed-Humid""","""None""","""Standard Efficiency""","""South Atlantic""","""South Atlantic""","""South""","""In another census Place""","""None""","""120% Usage""","""None""","""None""","""120% Usage""","""Electric Resistance""","""120% Usage""","""72F""","""No""","""0F""","""None""","""Double-Loaded Interior""","""G2400030""","""G2400030, G24001202""","""Anne Arundel County""","""None""","""290 Rated kWh""","""120% Usage""",…,"""None""","""MD""","""Renter""",1,"""High""",10,0.130352,"""0""","""3.219615385""","""11.25""","""1.217491908""","""0""","""3.224769231""","""Utility Rates - Fixed + Variab…","""data/simple_rates/State.tsv""","""Occupied""","""1980s""","""1980-99""","""Electric Standard""","""Electricity""","""No""","""None""","""Baltimore Washingto""",39.17,-76.68,"""F18 B18 L18 R18""","""Single, Clear, Non-metal""",185465,false,"""electrical_resistance""","""electrical_resistance""",true,false,false,false,false,true
0,252.301639,854,108692.0,"""CBSA Washington-Arlington-Alex…","""No""","""100-120%""","""4A""","""4A""","""Hour22""","""None""","""2""","""Mixed-Humid""","""None""","""None""","""South Atlantic""","""South Atlantic""","""South""","""In another census Place""","""None""","""100% Usage""","""None""","""None""","""100% Usage""","""Electric Resistance""","""100% Usage""","""68F""","""No""","""0F""","""None""","""Double-Loaded Interior""","""G2400330""","""G2400330, G24001102""","""Prince George's County""","""None""","""318 Rated kWh""","""100% Usage""",…,"""None""","""MD""","""Renter""",1,"""Medium""",10,0.130352,"""0""","""3.219615385""","""11.25""","""1.217491908""","""0""","""3.224769231""","""Utility Rates - Fixed + Variab…","""data/simple_rates/State.tsv""","""Occupied""","""1970s""","""1960-79""","""Electric Heat Pump, 50 gal, 3.…","""Electricity""","""No""","""None""","""Andrews Afb Camp Sp""",38.82,-76.85,"""

Simple EV charging model:
- Every household has 1 EV with the same battery capacity
- Every household has the same EV charger
- Every household has the same miles traveled (30 miles per day at 0.3 kWh/mile → needs to charge 9 kWh at night)
- Assume same electricity rate for everyone and it is constant by hour
- Every household does all their charging at home (no differentiation by hour because rates are held constant)

In [6]:
# EV Assumptions
EVS_PER_HOUSEHOLD = 1
DAILY_MILES = 30
KWH_PER_MILE = 0.30
DAYS_PER_YEAR = 365

daily_kwh_per_ev = DAILY_MILES * KWH_PER_MILE
annual_kwh_per_ev = daily_kwh_per_ev * DAYS_PER_YEAR

ev_model_simple = (
    metadata.select(
        "bldg_id",
        "weight",
        "in.county_name",
        "in.income",
        "in.puma_metro_status",
        "in.utility_bill_electricity_marginal_rates",
    )
    .rename({"in.utility_bill_electricity_marginal_rates": "rate_per_kwh"})
    .with_columns(
        pl.lit(EVS_PER_HOUSEHOLD).alias("ev_count"),
        pl.lit(daily_kwh_per_ev).alias("daily_ev_kwh"),
        pl.lit(annual_kwh_per_ev).alias("annual_ev_kwh_per_ev"),
    )
    .with_columns(
        (pl.col("annual_ev_kwh_per_ev") * pl.col("ev_count")).alias("annual_ev_kwh"),
    )
    .with_columns(
        (pl.col("annual_ev_kwh") * pl.col("rate_per_kwh")).alias("annual_ev_cost_usd"),
    )
    .with_columns(
        (pl.col("annual_ev_kwh") * pl.col("weight")).alias("weighted_annual_ev_kwh"),
        (pl.col("annual_ev_cost_usd") * pl.col("weight")).alias("weighted_annual_ev_cost_usd"),
    )
)

ev_model_simple.head(5)

bldg_id,weight,in.county_name,in.income,in.puma_metro_status,rate_per_kwh,ev_count,daily_ev_kwh,annual_ev_kwh_per_ev,annual_ev_kwh,annual_ev_cost_usd,weighted_annual_ev_kwh,weighted_annual_ev_cost_usd
i64,f64,str,str,str,f64,i32,f64,f64,f64,f64,f64,f64
185465,252.301639,"""Anne Arundel County""","""100000-119999""","""In metro area, not/partially i…",0.130352,1,9.0,3285.0,3285.0,428.207506,828810.883229,108037.455451
381889,252.301639,"""Prince George's County""","""100000-119999""","""In metro area, not/partially i…",0.130352,1,9.0,3285.0,3285.0,428.207506,828810.883229,108037.455451
294937,252.301639,"""Baltimore city""","""100000-119999""","""In metro area, principal city""",0.130352,1,9.0,3285.0,3285.0,428.207506,828810.883229,108037.455451
310780,252.301639,"""Baltimore County""","""<10000""","""In metro area, not/partially i…",0.130352,1,9.0,3285.0,3285.0,428.207506,828810.883229,108037.455451
313723,252.301639,"""Harford County""","""50000-59999""","""In metro area, not/partially i…",0.130352,1,9.0,3285.0,3285.0,428.207506,828810.883229,108037.455451


Slightly more complicated model:
- instead of assuming every household has one EV, predict the number of vehicles for each household using predict_num_vehicles() and assume they are all EVs
- sample trip schedules according to methods in ev_demand.py

To get this running, need to get data using command line arguments (PUMS is used to predict the number of vehicles per household):
- cd ~/buildstock-fetch
- uv sync --extra dev-utils          # scikit-learn for the ownership model
- just download-pums-md              # writes utils/ev_data/inputs/MD_2021_pums_PUMA_HINCP_VEH_NP.csv
- just download-nhts                 # for Phase 2 (trip schedules)

In [33]:
import sys
from datetime import datetime
from pathlib import Path

# buildstock-fetch EV modules (must come before rate-design-platform on sys.path)
BSF_ROOT = Path("/ebs/home/lily_switch_box/buildstock-fetch")
if str(BSF_ROOT) not in sys.path:
    sys.path.insert(0, str(BSF_ROOT))

from utils import ev_utils
from utils.ev_demand import EVDemandCalculator

PUMS_PATH = BSF_ROOT / "utils/ev_data/inputs/MD_2021_pums_PUMA_HINCP_VEH_NP.csv"
NHTS_PATH = BSF_ROOT / "utils/ev_data/inputs/NHTS_v2_1_trip_surveys.csv"

if not PUMS_PATH.exists():
    raise FileNotFoundError(
        f"Missing {PUMS_PATH}. Run: just download-pums-md"
    )
if not NHTS_PATH.exists():
    raise FileNotFoundError(
        f"Missing {NHTS_PATH}. Run: just download-nhts"
    )

# None = all MD buildings that pass load_metadata filters (~9,002); int = head(N) for fast iteration
N_VEH_SAMPLE: int | None = None

# load_metadata() is slim (vehicle-model columns only); keep full metadata for costs
metadata_slim = ev_utils.load_metadata(path_metadata, STATE)
if N_VEH_SAMPLE is not None:
    metadata_slim = metadata_slim.join(
        metadata.head(N_VEH_SAMPLE).select("bldg_id"), on="bldg_id"
    )
metadata_sample = metadata.join(metadata_slim.select("bldg_id"), on="bldg_id")
pums_df = ev_utils.load_pums_data(str(PUMS_PATH), path_metadata)

print(f"PUMS households: {pums_df.height:,}")
print(f"ResStock sample:   {metadata_slim.height:,}")
print(pums_df.group_by("vehicles").agg(pl.len().alias("n")).sort("vehicles"))

nhts_df = ev_utils.load_nhts_data(str(NHTS_PATH), STATE)
print(f"NHTS trips (MD census division): {nhts_df.height:,}")

PUMS households: 23,776
ResStock sample:   9,002
shape: (7, 2)
┌──────────┬──────┐
│ vehicles ┆ n    │
│ ---      ┆ ---  │
│ i64      ┆ u32  │
╞══════════╪══════╡
│ 0        ┆ 1578 │
│ 1        ┆ 7449 │
│ 2        ┆ 9240 │
│ 3        ┆ 3675 │
│ 4        ┆ 1302 │
│ 5        ┆ 382  │
│ 6        ┆ 150  │
└──────────┴──────┘
NHTS trips (MD census division): 5,352


In [34]:
SCHEDULE_START = datetime(2024, 1, 1)
SCHEDULE_END = datetime(2024, 12, 31)
EV_ADOPTION_RATE = 1.0  # fraction of predicted vehicles treated as EVs (assume 1 for now)

# create EV demand calculator
calculator = EVDemandCalculator(
    metadata_df=metadata_slim,
    nhts_df=nhts_df,  
    pums_df=pums_df,
    start_date=SCHEDULE_START,
    end_date=SCHEDULE_END,
    random_state=SEED,
)

bldg_vehicles = calculator.predict_num_vehicles()

print("Predicted vehicles per building")
print(bldg_vehicles.group_by("vehicles").agg(pl.len().alias("buildings")).sort("vehicles"))
print(f"Mean vehicles/household: {bldg_vehicles['vehicles'].mean():.2f}")

vehicle_profiles = calculator.sample_vehicle_profiles(bldg_vehicles, nhts_df)
print(f"Vehicle profiles sampled: {len(vehicle_profiles):,}")

# Peek at one profile
example_key = next(k for k, p in vehicle_profiles.items() if p.weekday_miles)
example = vehicle_profiles[example_key]
print(
    f"Example bldg {example.bldg_id} vehicle {example.vehicle_id}: "
    f"{len(example.weekday_miles)} weekday trips, "
    f"{len(example.weekend_miles)} weekend trips"
)

trip_schedules = calculator._generate_annual_trip_schedule(vehicle_profiles)
print(f"Trip schedule rows: {trip_schedules.height:,}")
trip_schedules.head(5)

2026-06-14 23:53:43,872 - INFO - Vehicle ownership model not fitted yet. Fitting model...
2026-06-14 23:53:43,911 - INFO - Sampling vehicle profiles for 9002 buildings...
2026-06-14 23:53:43,912 - INFO - Preparing NHTS data cache to optimize matching...
2026-06-14 23:53:43,915 - INFO - NHTS cache prepared successfully


Predicted vehicles per building
shape: (3, 2)
┌──────────┬───────────┐
│ vehicles ┆ buildings │
│ ---      ┆ ---       │
│ i64      ┆ u32       │
╞══════════╪═══════════╡
│ 0        ┆ 6         │
│ 1        ┆ 3022      │
│ 2        ┆ 5974      │
└──────────┴───────────┘
Mean vehicles/household: 1.66


2026-06-14 23:54:01,787 - INFO - Building progress: 9002/9002 (100.0%)
2026-06-14 23:54:01,788 - INFO - Generated 14970 vehicle profiles from 9002 buildings
2026-06-14 23:54:01,791 - INFO - Processing 14970 vehicle profiles...
2026-06-14 23:54:01,792 - INFO - Using parallel processing with all available workers


Vehicle profiles sampled: 14,970
Example bldg 185465 vehicle 1: 2 weekday trips, 0 weekend trips


2026-06-14 23:59:19,077 - INFO - Progress: 10000/14970 (66.8%)
2026-06-15 00:01:56,004 - INFO - Progress: 14970/14970 (100.0%)
2026-06-15 00:01:56,008 - INFO - Generated 12087360 trip schedules from 14970 vehicle profiles


Trip schedule rows: 12,087,360


bldg_id,vehicle_id,date,departure_hour,arrival_hour,miles_driven
i64,i64,datetime[μs],i64,i64,f64
389682,1,2024-01-06 00:00:00,19,19,8.495297
389682,1,2024-01-06 00:00:00,19,19,8.624196
389682,1,2024-01-07 00:00:00,19,21,9.170579
389682,1,2024-01-07 00:00:00,19,18,8.014251
389682,1,2024-01-13 00:00:00,19,19,8.448941


In [35]:
bldg_annual_miles = (
    trip_schedules.group_by("bldg_id")
    .agg(
        pl.col("miles_driven").sum().alias("annual_miles"),
        pl.col("miles_driven").mean().alias("mean_miles_per_trip"),
        pl.len().alias("trip_count"),
    )
)

ev_model_nhts = (
    metadata_sample.select(
        "bldg_id",
        "weight",
        "in.county_name",
        "in.income",
        "in.utility_bill_electricity_marginal_rates",
    )
    .join(bldg_vehicles.select("bldg_id", "vehicles"), on="bldg_id")
    .join(bldg_annual_miles, on="bldg_id", how="left")
    .with_columns(
        pl.col("annual_miles").fill_null(0.0),
        pl.col("trip_count").fill_null(0),
    )
    .rename({"in.utility_bill_electricity_marginal_rates": "rate_per_kwh"})
    .with_columns(
        (pl.col("vehicles") * EV_ADOPTION_RATE).alias("ev_count"),
    )
    .with_columns(
        (pl.col("annual_miles") * KWH_PER_MILE * EV_ADOPTION_RATE).alias("annual_ev_kwh"),
    )
    .with_columns(
        (pl.col("annual_ev_kwh") / DAYS_PER_YEAR).alias("daily_ev_kwh"),
        (pl.col("annual_ev_kwh") / pl.col("ev_count").clip(lower_bound=1)).alias("annual_ev_kwh_per_ev"),
    )
    .with_columns(
        (pl.col("annual_ev_kwh") * pl.col("rate_per_kwh")).alias("annual_ev_cost_usd"),
    )
    .with_columns(
        (pl.col("annual_ev_kwh") * pl.col("weight")).alias("weighted_annual_ev_kwh"),
        (pl.col("annual_ev_cost_usd") * pl.col("weight")).alias("weighted_annual_ev_cost_usd"),
    )
)

print("Miles per building (sample)")
print(
    ev_model_nhts.select(
        pl.col("annual_miles").mean().alias("mean"),
        pl.col("annual_miles").median().alias("median"),
        pl.col("annual_miles").max().alias("max"),
    )
)
print(f"Flat-assumption benchmark: {DAILY_MILES * DAYS_PER_YEAR:.0f} mi/EV/year")

print("\nComparison (same buildings, unweighted annual_ev_kwh totals)")
sample_ids = metadata_sample.select("bldg_id")
simple_on_sample = ev_model_simple.join(sample_ids, on="bldg_id")
pums_flat_on_sample = (
    simple_on_sample.drop("ev_count", "daily_ev_kwh", "annual_ev_kwh_per_ev", "annual_ev_kwh", "annual_ev_cost_usd", "weighted_annual_ev_kwh", "weighted_annual_ev_cost_usd")
    .join(bldg_vehicles.select("bldg_id", "vehicles"), on="bldg_id")
    .with_columns((pl.col("vehicles") * EV_ADOPTION_RATE * annual_kwh_per_ev).alias("annual_ev_kwh"))
)
print(f"  Simple (1 EV, flat miles):    {simple_on_sample['annual_ev_kwh'].sum():,.0f} kWh")
print(f"  PUMS counts, flat miles:      {pums_flat_on_sample['annual_ev_kwh'].sum():,.0f} kWh")
print(f"  PUMS + NHTS trip schedules:   {ev_model_nhts['annual_ev_kwh'].sum():,.0f} kWh")

ev_model_nhts.head(10)

Miles per building (sample)
shape: (1, 3)
┌─────────────┬────────────┬───────────────┐
│ mean        ┆ median     ┆ max           │
│ ---         ┆ ---        ┆ ---           │
│ f64         ┆ f64        ┆ f64           │
╞═════════════╪════════════╪═══════════════╡
│ 14336.30104 ┆ 8803.02479 ┆ 291893.632854 │
└─────────────┴────────────┴───────────────┘
Flat-assumption benchmark: 10950 mi/EV/year

Comparison (same buildings, unweighted annual_ev_kwh totals)
  Simple (1 EV, flat miles):    29,571,570 kWh
  PUMS counts, flat miles:      49,176,450 kWh
  PUMS + NHTS trip schedules:   38,716,615 kWh


bldg_id,weight,in.county_name,in.income,rate_per_kwh,vehicles,annual_miles,mean_miles_per_trip,trip_count,ev_count,annual_ev_kwh,daily_ev_kwh,annual_ev_kwh_per_ev,annual_ev_cost_usd,weighted_annual_ev_kwh,weighted_annual_ev_cost_usd
i64,f64,str,str,f64,i64,f64,f64,u32,f64,f64,f64,f64,f64,f64,f64
185465,252.301639,"""Anne Arundel County""","""100000-119999""",0.130352,2,10950.941484,11.649938,940,2.0,3285.282445,9.000774,1642.641223,428.244323,828882.144599,108046.744539
381889,252.301639,"""Prince George's County""","""100000-119999""",0.130352,2,14857.930183,9.45161,1572,2.0,4457.379055,12.211997,2228.689528,581.029884,1.1246e6,146594.791808
294937,252.301639,"""Baltimore city""","""100000-119999""",0.130352,1,7727.493734,14.747125,524,1.0,2318.24812,6.351365,2318.24812,302.189116,584897.799739,76242.80914
310780,252.301639,"""Baltimore County""","""<10000""",0.130352,1,328.743203,0.627373,524,1.0,98.622961,0.2702,98.622961,12.855736,24882.734619,3243.523206
313723,252.301639,"""Harford County""","""50000-59999""",0.130352,1,2368.391222,2.259915,1048,1.0,710.517367,1.946623,710.517367,92.617616,179264.695974,23367.576364
389682,252.301639,"""Baltimore County""","""45000-49999""",0.130352,1,1699.300314,8.169713,208,1.0,509.790094,1.396685,509.790094,66.452342,128620.876192,16766.034886
420703,252.301639,"""Howard County""","""30000-34999""",0.130352,1,1158.572905,2.211017,524,1.0,347.571872,0.952252,347.571872,45.306814,87692.95278,11430.983438
421768,252.301639,"""Anne Arundel County""","""200000+""",0.130352,2,18594.16627,3.942783,4716,2.0,5578.249881,15.282876,2789.124941,727.138042,1.4074e6,183458.119641
429764,252.301639,"""Montgomery County""","""180000-199999""",0.130352,2,8747.979269,13.929903,628,2.0,2624.393781,7.19012,1312.19689,342.095926,662138.851558,86311.36261


In [8]:
metadata['in.electric_vehicle'].value_counts()

in.electric_vehicle,count
str,u32
"""None""",9996


In [5]:
# 1. Keyword search
patterns = ["vehicle", "ev", "plug", "charge", "utility", "income", "occup", "puma", "county", "weight"]
for pat in patterns:
    hits = [c for c in metadata.columns if pat.lower() in c.lower()]
    if hits:
        print(f"\n--- {pat} ---")
        print(hits)

# 2. Prefix convention
in_cols  = [c for c in metadata.columns if c.startswith("in.")]
out_cols = [c for c in metadata.columns if c.startswith("out.")]
sb_cols  = [c for c in metadata.columns if c.startswith("sb.")]

# 3. Inspect a column
metadata.group_by("in.tenure").agg(pl.len(), pl.col("weight").sum())

# 4. Hourly load endpoints (from a sample parquet)
import glob
sample = pl.read_parquet(glob.glob(f"{path_loads_dir}/*.parquet")[0], n_rows=1)
[c for c in sample.columns if "electricity" in c]


--- vehicle ---
['in.electric_vehicle']

--- ev ---
['in.clothes_dryer_usage_level', 'in.clothes_washer_usage_level', 'in.cooking_range_usage_level', 'in.dishwasher_usage_level', 'in.federal_poverty_level', 'in.geometry_building_level_mf', 'in.refrigerator_usage_level', 'in.usage_level']

--- plug ---
['in.plug_load_diversity', 'in.plug_loads']

--- charge ---
['in.hvac_system_single_speed_ac_charge', 'in.hvac_system_single_speed_ashp_charge', 'in.utility_bill_electricity_fixed_charges', 'in.utility_bill_fuel_oil_fixed_charges', 'in.utility_bill_natural_gas_fixed_charges', 'in.utility_bill_propane_fixed_charges']

--- utility ---
['in.utility_bill_electricity_fixed_charges', 'in.utility_bill_electricity_marginal_rates', 'in.utility_bill_fuel_oil_fixed_charges', 'in.utility_bill_fuel_oil_marginal_rates', 'in.utility_bill_natural_gas_fixed_charges', 'in.utility_bill_natural_gas_marginal_rates', 'in.utility_bill_propane_fixed_charges', 'in.utility_bill_propane_marginal_rates', 'in.utilit

['out.electricity.ceiling_fan.energy_consumption',
 'out.electricity.ceiling_fan.energy_consumption_intensity',
 'out.electricity.clothes_dryer.energy_consumption',
 'out.electricity.clothes_dryer.energy_consumption_intensity',
 'out.electricity.clothes_washer.energy_consumption',
 'out.electricity.clothes_washer.energy_consumption_intensity',
 'out.electricity.cooling.energy_consumption',
 'out.electricity.cooling.energy_consumption_intensity',
 'out.electricity.cooling_fans_pumps.energy_consumption',
 'out.electricity.cooling_fans_pumps.energy_consumption_intensity',
 'out.electricity.dishwasher.energy_consumption',
 'out.electricity.dishwasher.energy_consumption_intensity',
 'out.electricity.freezer.energy_consumption',
 'out.electricity.freezer.energy_consumption_intensity',
 'out.electricity.heating.energy_consumption',
 'out.electricity.heating.energy_consumption_intensity',
 'out.electricity.heating_fans_pumps.energy_consumption',
 'out.electricity.heating_fans_pumps.energy_cons